In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import test_transforms
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir)
transformed_dataset = ImageDataset(annot_path, img_dir, test_transforms)

In [3]:
_, truth_labels = next(iter(transformed_dataset))
truth_labels.shape, truth_labels

(torch.Size([5, 5]),
 tensor([[8.0000, 0.5848, 0.7321, 0.1205, 0.3393],
         [8.0000, 0.4196, 0.8482, 0.1741, 0.2902],
         [8.0000, 0.0714, 0.8259, 0.1250, 0.3482],
         [8.0000, 0.5357, 0.6562, 0.1071, 0.2812],
         [8.0000, 0.5893, 0.5402, 0.0714, 0.0893]]))

In [4]:
bboxes = truth_labels[:, -4:]
bboxes.shape, bboxes

(torch.Size([5, 4]),
 tensor([[0.5848, 0.7321, 0.1205, 0.3393],
         [0.4196, 0.8482, 0.1741, 0.2902],
         [0.0714, 0.8259, 0.1250, 0.3482],
         [0.5357, 0.6562, 0.1071, 0.2812],
         [0.5893, 0.5402, 0.0714, 0.0893]]))

In [5]:
bboxes[..., [0, 2]] *= 224
bboxes[..., [1, 3]] *= 224

In [6]:
x, y, w, h = bboxes.unbind(-1)
x, y, w, h

(tensor([131.,  94.,  16., 120., 132.]),
 tensor([164.0000, 190.0000, 185.0000, 147.0000, 121.0000]),
 tensor([27., 39., 28., 24., 16.]),
 tensor([76., 65., 78., 63., 20.]))

In [12]:
xyxy = torch.stack([
    x - w / 2,
    y - h / 2,
    x + w / 2,
    y + h / 2
], dim=-1)

xyxy, xyxy.shape

(tensor([[117.5000, 126.0000, 144.5000, 202.0000],
         [ 74.5000, 157.5000, 113.5000, 222.5000],
         [  2.0000, 146.0000,  30.0000, 224.0000],
         [108.0000, 115.5000, 132.0000, 178.5000],
         [124.0000, 111.0000, 140.0000, 131.0000]]),
 torch.Size([5, 4]))

In [ ]:
class_preds = torch.randn(100, 21)

In [ ]:
import torch.nn.functional as F

class_costs = [-F.softmax(class_preds, dim=-1).select(-1, idx).unsqueeze(-1)
                   for idx in range(truth_labels.shape[0])]

In [ ]:
len(class_costs)

In [ ]:
torch.cat(class_costs, dim=-1).shape

In [ ]:
bbox_preds = torch.randn(100, 4)
bbox_preds[0], truth_labels[0][1:]

In [ ]:
l1_costs = [(bbox_preds - truth_labels[i][1:]).abs().sum(-1, True) for i in range(truth_labels.shape[0])]
torch.cat(l1_costs, dim=-1).shape

In [ ]:
l1_costs

In [ ]:
subtracted = (bbox_preds - truth_labels[0][1:])
subtracted.shape, subtracted

In [ ]:
absoluted = subtracted.abs()
absoluted

In [ ]:
summed = absoluted.sum(dim=-1)
summed

In [ ]:
summed.unsqueeze(-1).shape